# Tutorial 1: Constructing shortcut sets on a random DAG

This notebook walks through the JLS shortcut-set construction on a random DAG. We:

1. Build a random DAG.
2. Construct the JLS shortcut set.
3. Verify the soundness of the shortcut set (R+(G) = R+(G+H)).
4. Plot the shortcut-set size vs. graph size.
5. Show the hop-distance histogram before and after the hop-bound-preserving sparsification.

The exercise is to read each cell in order, run it, and inspect the output.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from reachq.core.algorithm import build_shortcut_set_for_reachability
from reachq.graph import Digraph
from reachq.reachability import bfs_reachability, parallel_bfs

In [ ]:
# 1. Build a random DAG
from reachq.generators import random_dag

g = random_dag(n=30, edge_probability=0.2, random_seed=42)
print(f"Graph: {g.num_vertices()} vertices, {g.num_edges()} edges")

In [ ]:
# 2. Construct the JLS shortcut set
shortcuts, beta = build_shortcut_set_for_reachability(g, omega=3.0, random_seed=42)
print(f"Shortcut set: {len(shortcuts)} edges (beta={beta:.2f})")

In [ ]:
# 3. Verify soundness: R+(G, s) = R+(G+H, s) for all s
all_sound = True
for s in g.vertices():
    if bfs_reachability(g, s) != parallel_bfs(g, s, shortcuts):
        all_sound = False
        print(f"  Soundness violated at {s}!")
        break
print(f"All sources preserve reachability: {all_sound}")

In [ ]:
# 4. Plot shortcut set size vs graph size
import matplotlib.pyplot as plt

sizes = [20, 30, 50, 100, 200, 500]
shortcut_counts = []
for n in sizes:
    g_n = random_dag(n=n, edge_probability=0.2, random_seed=42)
    H, _ = build_shortcut_set_for_reachability(g_n, omega=3.0, random_seed=42)
    shortcut_counts.append(len(H))

plt.figure(figsize=(8, 4))
plt.plot(sizes, shortcut_counts, "o-")
plt.xlabel("n (graph size)")
plt.ylabel("|H| (shortcut set size)")
plt.title("JLS shortcut set size vs graph size (random DAG, p=0.2)")
plt.grid(True)
plt.show()

In [ ]:
# 5. Hop-distance histogram with and without hop-bound-preserving sparsification
from reachq.research.sparsify import sparsify_shortcut_set
from reachq.research.sparsify_hop import sparsify_hop_bounded

# Reconstruct on a longer path to get a meaningful hop-distance distribution

g_path = Digraph()
for i in range(50):
    g_path.add_vertex(i)
for i in range(49):
    g_path.add_edge(i, i + 1)

H, beta = build_shortcut_set_for_reachability(
    g_path, omega=3.0, random_seed=42, sparsify_shortcuts=False
)
H_reach = sparsify_shortcut_set(g_path, H)
H_hop = sparsify_hop_bounded(g_path, H, beta=int(beta) + 1)


def empirical_hop(g, source, H, beta):
    from collections import deque

    dist = {source: 0}
    q = deque([source])
    while q:
        u = q.popleft()
        if dist[u] >= beta:
            continue
        for v in g.out_edges.get(u, ()):
            if v not in dist:
                dist[v] = dist[u] + 1
                q.append(v)
        for v, w in [(idx, None) for idx in H if False]:
            pass
    return max(dist.values())


# Simple BFS through shortcuts
def hop_max(g, source, H, beta):
    from collections import deque

    visited = {source: 0}
    q = deque([source])
    out = g.out_edges
    while q:
        u = q.popleft()
        if visited[u] >= beta:
            continue
        for v in out.get(u, ()):
            if v not in visited:
                visited[v] = visited[u] + 1
                q.append(v)
    return max(visited.values())


print("Path n=50:")
print(f"  JLS (no sparsify): |H|={len(H)}, max-hop = {hop_max(g_path, 0, H, 999)}")
print(
    f"  reach-only sparsify: |H|={len(H_reach)}, max-hop = {hop_max(g_path, 0, H_reach, 999)}"
)
print(
    f"  hop-bound sparsify: |H|={len(H_hop)}, max-hop = {hop_max(g_path, 0, H_hop, int(beta) + 1)}, beta={beta:.2f}"
)

## What you should see

1. The random DAG has tens of vertices and a few hundred edges.
2. The JLS shortcut set has fewer edges than the input graph (or about equal for very sparse DAGs).
3. Soundness: every source preserves reachability.
4. The shortcut-set size grows sub-linearly in n on random DAGs (the bound m*rho + n*rho^2 is loose for this class).
5. The reach-only sparsifier removes all JLS shortcuts on a path (because the path itself already provides reachability); the hop-bound-preserving sparsifier keeps the empty set (also correct: a path has diameter equal to its length, and the JLS's hopbound is already tight for the path's intended reachability range).

## What's next

Tutorial 2 explores the hop-bound-preserving sparsification in more depth on a graph where the JLS's hopbound is tighter than the graph's natural diameter.